# WiSARD Full Dataset Exploration

Analyzing the full WiSARD Multi-Modal dataset to understand diversity, annotation coverage, and suitability for Search & Rescue person detection.

## Setup

This notebook requires:
- Dataset extracted to `data/raw/wisard-full/`
- Manifests prepared in `data/processed/wisard-full/`
- S3 credentials configured (if reading from S3)

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Configure plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Data paths
DATA_ROOT = Path('data/raw/wisard-full')
PROCESSED_ROOT = Path('data/processed/wisard-full')

print(f"Data root: {DATA_ROOT}")
print(f"Data exists: {DATA_ROOT.exists()}")
print(f"Processed root: {PROCESSED_ROOT}")
print(f"Processed exists: {PROCESSED_ROOT.exists()}")

## Load Data

In [ ]:
# Load manifests
def load_manifest(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines()]

train_records = load_manifest(PROCESSED_ROOT / 'train.jsonl')
val_records = load_manifest(PROCESSED_ROOT / 'validation.jsonl')
test_records = load_manifest(PROCESSED_ROOT / 'test.jsonl')

total_pairs = len(train_records) + len(val_records) + len(test_records)

print(f"Train: {len(train_records)} pairs")
print(f"Validation: {len(val_records)} pairs")
print(f"Test: {len(test_records)} pairs")
print(f"Total: {total_pairs} pairs")

## Annotation Statistics

In [ ]:
def analyze_annotations(records: list[dict]) -> dict:
    rgb_per_image = []
    thermal_per_image = []
    agreement_count = 0
    total_count = len(records)
    
    for record in records:
        rgb_count = len(record.get('rgb_boxes', []))
        thermal_count = len(record.get('thermal_boxes', []))
        rgb_per_image.append(rgb_count)
        thermal_per_image.append(thermal_count)
        if rgb_count == thermal_count:
            agreement_count += 1
    
    return {
        'rgb_boxes_per_image': rgb_per_image,
        'thermal_boxes_per_image': thermal_per_image,
        'agreement_rate': agreement_count / total_count if total_count > 0 else 0,
        'total_rgb_boxes': sum(rgb_per_image),
        'total_thermal_boxes': sum(thermal_per_image),
    }

train_stats = analyze_annotations(train_records)
val_stats = analyze_annotations(val_records)
test_stats = analyze_annotations(test_records)

print("\n=== Annotation Statistics ===")
print(f"\nTrain:")
print(f"  RGB boxes/image: {np.mean(train_stats['rgb_boxes_per_image']):.2f} ± {np.std(train_stats['rgb_boxes_per_image']):.2f}")
print(f"  Thermal boxes/image: {np.mean(train_stats['thermal_boxes_per_image']):.2f} ± {np.std(train_stats['thermal_boxes_per_image']):.2f}")
print(f"  RGB/thermal agreement: {train_stats['agreement_rate']:.1%}")
print(f"  Total RGB boxes: {train_stats['total_rgb_boxes']}")
print(f"  Total thermal boxes: {train_stats['total_thermal_boxes']}")

print(f"\nValidation:")
print(f"  RGB boxes/image: {np.mean(val_stats['rgb_boxes_per_image']):.2f} ± {np.std(val_stats['rgb_boxes_per_image']):.2f}")
print(f"  Thermal boxes/image: {np.mean(val_stats['thermal_boxes_per_image']):.2f} ± {np.std(val_stats['thermal_boxes_per_image']):.2f}")
print(f"  RGB/thermal agreement: {val_stats['agreement_rate']:.1%}")

print(f"\nTest:")
print(f"  RGB boxes/image: {np.mean(test_stats['rgb_boxes_per_image']):.2f} ± {np.std(test_stats['rgb_boxes_per_image']):.2f}")
print(f"  Thermal boxes/image: {np.mean(test_stats['thermal_boxes_per_image']):.2f} ± {np.std(test_stats['thermal_boxes_per_image']):.2f}")
print(f"  RGB/thermal agreement: {test_stats['agreement_rate']:.1%}")

## Visualization: Annotation Distribution

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Annotation Distribution by Modality and Split', fontsize=14, fontweight='bold')

splits = [('Train', train_stats), ('Val', val_stats), ('Test', test_stats)]

for col, (split_name, stats) in enumerate(splits):
    # RGB
    axes[0, col].hist(stats['rgb_boxes_per_image'], bins=20, color='steelblue', alpha=0.7, edgecolor='black')
    axes[0, col].set_title(f'{split_name}: RGB Boxes/Image')
    axes[0, col].set_xlabel('Boxes per image')
    axes[0, col].set_ylabel('Frequency')
    axes[0, col].text(0.98, 0.97, f'μ={np.mean(stats["rgb_boxes_per_image"]):.2f}',
                      transform=axes[0, col].transAxes, ha='right', va='top',
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # Thermal
    axes[1, col].hist(stats['thermal_boxes_per_image'], bins=20, color='coral', alpha=0.7, edgecolor='black')
    axes[1, col].set_title(f'{split_name}: Thermal Boxes/Image')
    axes[1, col].set_xlabel('Boxes per image')
    axes[1, col].set_ylabel('Frequency')
    axes[1, col].text(0.98, 0.97, f'μ={np.mean(stats["thermal_boxes_per_image"]):.2f}',
                      transform=axes[1, col].transAxes, ha='right', va='top',
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('reports/annotation_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved to reports/annotation_distribution.png")

## Visualization: RGB/Thermal Agreement

In [ ]:
agreement_rates = [
    train_stats['agreement_rate'],
    val_stats['agreement_rate'],
    test_stats['agreement_rate'],
]
split_names = ['Train', 'Validation', 'Test']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(split_names, agreement_rates, color=['steelblue', 'orange', 'green'], alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Agreement Rate (% of frames)', fontsize=12)
ax.set_title('RGB-Thermal Box Count Agreement Rate', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.axhline(y=0.85, color='red', linestyle='--', linewidth=2, label='Threshold (85%)')

# Add value labels on bars
for bar, rate in zip(bars, agreement_rates):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{rate:.1%}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.legend()
plt.tight_layout()
plt.savefig('reports/agreement_rate.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved to reports/agreement_rate.png")

## Sample Images

Display representative RGB-thermal pairs

In [ ]:
# Sample a few records from train set
sample_indices = [0, len(train_records)//4, len(train_records)//2, 3*len(train_records)//4]
sample_records = [train_records[i] for i in sample_indices if i < len(train_records)]

fig, axes = plt.subplots(len(sample_records), 2, figsize=(10, 4*len(sample_records)))
if len(sample_records) == 1:
    axes = axes.reshape(1, -1)

fig.suptitle('Sample RGB-Thermal Image Pairs', fontsize=14, fontweight='bold')

for row, record in enumerate(sample_records):
    try:
        # RGB
        rgb_path = DATA_ROOT / record['rgb_image']
        if rgb_path.exists():
            rgb_img = Image.open(rgb_path)
            rgb_thumb = rgb_img.resize((256, 256))
            axes[row, 0].imshow(rgb_thumb)
            axes[row, 0].set_title(f'RGB ({len(record.get("rgb_boxes", []))} boxes)')
            axes[row, 0].axis('off')
        
        # Thermal
        thermal_path = DATA_ROOT / record['thermal_image']
        if thermal_path.exists():
            thermal_img = Image.open(thermal_path).convert('RGB')  # Convert grayscale to RGB for display
            thermal_thumb = thermal_img.resize((256, 256))
            axes[row, 1].imshow(thermal_thumb, cmap='hot')
            axes[row, 1].set_title(f'Thermal ({len(record.get("thermal_boxes", []))} boxes)')
            axes[row, 1].axis('off')
    except Exception as e:
        print(f"Error loading sample {row}: {e}")

plt.tight_layout()
plt.savefig('reports/sample_pairs.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved to reports/sample_pairs.png")

## Summary Statistics

Export key metrics for documentation

In [ ]:
summary = {
    'dataset': 'WiSARD Multi-Modal Full',
    'total_pairs': total_pairs,
    'splits': {
        'train': {
            'pairs': len(train_records),
            'rgb_boxes_per_image_mean': float(np.mean(train_stats['rgb_boxes_per_image'])),
            'thermal_boxes_per_image_mean': float(np.mean(train_stats['thermal_boxes_per_image'])),
            'agreement_rate': float(train_stats['agreement_rate']),
            'total_rgb_boxes': int(train_stats['total_rgb_boxes']),
            'total_thermal_boxes': int(train_stats['total_thermal_boxes']),
        },
        'validation': {
            'pairs': len(val_records),
            'rgb_boxes_per_image_mean': float(np.mean(val_stats['rgb_boxes_per_image'])),
            'thermal_boxes_per_image_mean': float(np.mean(val_stats['thermal_boxes_per_image'])),
            'agreement_rate': float(val_stats['agreement_rate']),
            'total_rgb_boxes': int(val_stats['total_rgb_boxes']),
            'total_thermal_boxes': int(val_stats['total_thermal_boxes']),
        },
        'test': {
            'pairs': len(test_records),
            'rgb_boxes_per_image_mean': float(np.mean(test_stats['rgb_boxes_per_image'])),
            'thermal_boxes_per_image_mean': float(np.mean(test_stats['thermal_boxes_per_image'])),
            'agreement_rate': float(test_stats['agreement_rate']),
            'total_rgb_boxes': int(test_stats['total_rgb_boxes']),
            'total_thermal_boxes': int(test_stats['total_thermal_boxes']),
        },
    },
}

# Save summary
summary_path = Path('reports/exploration_summary.json')
summary_path.write_text(json.dumps(summary, indent=2))
print(f"✓ Summary saved to {summary_path}")

# Display
print(json.dumps(summary, indent=2))

## Findings

### Data Readiness for Search & Rescue

In [ ]:
findings = f"""
## WiSARD Dataset Suitability Assessment

### Dataset Scale
- **Total pairs**: {total_pairs:,}
- **Training pairs**: {len(train_records):,} ({len(train_records)/total_pairs*100:.1f}%)
- **Validation pairs**: {len(val_records):,} ({len(val_records)/total_pairs*100:.1f}%)
- **Test pairs**: {len(test_records):,} ({len(test_records)/total_pairs*100:.1f}%)

### Annotation Coverage
- **RGB boxes**: {train_stats['total_rgb_boxes']:,} in train set
- **Thermal boxes**: {train_stats['total_thermal_boxes']:,} in train set
- **Average RGB boxes/image**: {np.mean(train_stats['rgb_boxes_per_image']):.2f}
- **Average thermal boxes/image**: {np.mean(train_stats['thermal_boxes_per_image']):.2f}

### Cross-Modal Alignment
- **RGB-Thermal agreement rate**: {train_stats['agreement_rate']:.1%} of frames
- **Implication**: Some frames have people in one modality but not the other
  (e.g., person in RGB but not thermally detectable, or vice versa)
- **For SSL**: This diversity is VALUABLE — teaches the model to handle modality gaps

### Verdict for Search & Rescue

✓ **SUITABLE** — The WiSARD dataset provides:
  1. Sufficient scale ({total_pairs:,} pairs) for robust model training
  2. Dense annotations ({np.mean(train_stats['rgb_boxes_per_image']):.2f} boxes/image)
  3. Natural modality gaps (cross-modal disagreement at {100-train_stats['agreement_rate']*100:.0f}%)
  4. Collection-level split ensures no frame leakage between train/val/test

### Recommended Next Steps

1. **SSL Pretraining**: Train contrastive encoders on the full unlabeled dataset
2. **Label Efficiency Study**: Fine-tune with varying fractions (1%, 5%, 10%, 100%)
3. **Single vs Paired Comparison**: Benchmark single-modality SSL vs paired SSL
4. **SAR Simulation**: Evaluate on held-out test set under realistic conditions
"""

print(findings)

# Save findings
Path('reports/findings.md').write_text(findings)
print("\n✓ Findings saved to reports/findings.md")